In [15]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy
from functools import lru_cache

In [16]:
np.random.seed(0)

In [17]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [18]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [19]:
def manipulation_thresholds(thresholds, priors, c):
    if np.sum(priors) == 0:
        return thresholds.copy()
    manip_thresholds = np.maximum(0, thresholds - (bayesian_update(priors) / c))
    return manip_thresholds

In [20]:
def merge_classifiers(thresholds, priors, manip_thresholds):
    if len(thresholds) <= 1:
        return thresholds, priors, manip_thresholds
    
    merged_thresholds = [thresholds[-1]]
    merged_priors = [priors[-1]]
    merged_manip_thresholds = [manip_thresholds[-1]]

    for i in range(len(thresholds)-2,-1,-1):
        if manip_thresholds[i] < merged_manip_thresholds[-1]:
            merged_thresholds.append(thresholds[i])
            merged_priors.append(priors[i])
            merged_manip_thresholds.append(manip_thresholds[i])
        else:
            merged_priors[-1] += priors[i]
    
    return np.array(merged_thresholds)[::-1], np.array(merged_priors)[::-1], np.array(merged_manip_thresholds)[::-1]

In [21]:
def accuracy_loss(thresholds, priors, manip_thresholds, threshold_true):
    losses = []
    for i in range(len(thresholds)):
        loss = np.abs(manip_thresholds[i] - threshold_true)
        losses.append(loss)
    losses = np.array(losses)
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [22]:
def evaluate_partition(partition, thresholds, priors, threshold_true, c, return_all=False):
    thresholds_p = thresholds[partition]
    priors_p = priors[partition]
    manip_thresholds_p = manipulation_thresholds(thresholds_p, priors_p, c)
    thresholds_p, priors_p, manip_thresholds_p = merge_classifiers(thresholds_p, priors_p, manip_thresholds_p)
    acc_loss_p = accuracy_loss(thresholds_p, priors_p, manip_thresholds_p, threshold_true)
    if return_all:
        return acc_loss_p, thresholds_p, priors_p, manip_thresholds_p
    return acc_loss_p

def evaluate_system(partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        acc_loss_p = evaluate_partition(partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [23]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [24]:
def display_queue(Q, P):
    res = "[  "
    for (a_id, b_id) in Q:
        res += f"({P[a_id]}, {P[b_id]})  "
    res += "]"
    print(res)


def find_partitions_greedy_lex(thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1

    Q = collections.deque(itertools.combinations(P.keys(), 2))
    while Q:
        if display:
            display_queue(Q, P)
        a_id, b_id = Q.popleft()
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        acc_loss_ab = evaluate_partition(ab, thresholds, priors, threshold_true, c)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    Q.append((new_id, p_id))

            Q = collections.deque(sorted(Q, key=lambda x: P[x[0]]))
            
    return list(P.values())

In [25]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({acc_loss:.4f}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)


def find_partitions_greedy_best(thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = block
        next_id += 1
    
    Q = collections.deque(itertools.combinations(P.keys(), 2))
    pq = []
    for a_id, b_id in Q:
        acc_loss_ab = evaluate_partition(sorted(P[a_id]+P[b_id]), thresholds, priors, threshold_true, c)
        heapq.heappush(pq, (acc_loss_ab, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        acc_loss_ab, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        acc_loss_a = evaluate_partition(a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(b, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])

        ab = sorted(a + b)
        rhs = acc_loss_ab * np.sum(priors[ab])

        if lhs > rhs:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    acc_loss = evaluate_partition(sorted(P[x_id]+P[y_id]), thresholds, priors, threshold_true, c)
                    heapq.heappush(pq2, (acc_loss, (x_id,y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    acc_loss = evaluate_partition(sorted(P[p_id]+P[new_id]), thresholds, priors, threshold_true, c)
                    heapq.heappush(pq, (acc_loss, (new_id, p_id)))
    return list(P.values())

In [26]:
def find_partitions_optimal(thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))

    @lru_cache(maxsize=None)
    def evaluate_partition_cached(partition_tuple):
        partition = list(partition_tuple)
        acc_loss_p = evaluate_partition(
            partition, thresholds, priors, threshold_true, c
        )
        return acc_loss_p * np.sum(priors[partition])

    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.0
        for partition in partitions:
            partition_tuple = tuple(sorted(partition))
            acc_loss += evaluate_partition_cached(partition_tuple)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = deepcopy(partitions)

    return best_partition

In [27]:
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
# thresholds = np.array([0.2, 0.4, 0.6, 0.8])

# priors = np.array([1/4, 1/4, 1/4, 1/4])
threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)

# priors = np.zeros_like(thresholds)
# balance_priors(priors, random=True)
priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])

In [28]:
C = np.arange(0.1, 20, 0.1)
# C = [10]

regularizer = []
accuracy_losses_greedy_lex = []
accuracy_losses_greedy_best = []
accuracy_losses_optimal = []

for c in tqdm.tqdm(C):
    partition_greedy_lex = find_partitions_greedy_lex(thresholds, priors, threshold_true, c, len(C)==1)
    partition_greedy_best = find_partitions_greedy_best(thresholds, priors, threshold_true, c, len(C)==1)
    partition_optimal = find_partitions_optimal(thresholds, priors, threshold_true, c)

    acc_loss_greedy_lex = evaluate_system(partition_greedy_lex, thresholds, priors, threshold_true, c)
    acc_loss_greedy_best = evaluate_system(partition_greedy_best, thresholds, priors, threshold_true, c)
    acc_loss_optimal = evaluate_system(partition_optimal, thresholds, priors, threshold_true, c)

    regularizer.append(c)
    accuracy_losses_greedy_lex.append(acc_loss_greedy_lex.item())
    accuracy_losses_greedy_best.append(acc_loss_greedy_best.item())
    accuracy_losses_optimal.append(acc_loss_optimal.item())

100%|██████████| 199/199 [00:09<00:00, 21.63it/s]


In [29]:
ratio_lex, ratio_best = [], []
for i in range(len(C)):
    ratio_lex.append((1 - accuracy_losses_optimal[i]) / (1 - accuracy_losses_greedy_lex[i]))
    ratio_best.append((1 - accuracy_losses_optimal[i]) / (1 - accuracy_losses_greedy_best[i]))

In [31]:
results = {"c": regularizer, "greedy_lex": accuracy_losses_greedy_lex, "greedy_best": accuracy_losses_greedy_best, "optimal": accuracy_losses_optimal}
px.line(results, x="c", y=["greedy_lex", "greedy_best", "optimal"], markers=len(C)==1).update_layout(yaxis=dict(title="Accuracy Loss"))

In [32]:
results_ratio = {"c": C, "ratio_lex": ratio_lex, "ratio_best": ratio_best}
px.line(results_ratio, x="c", y=["ratio_lex", "ratio_best"], markers=len(C)==1)